In [1]:
import pandas as pd
import numpy as np

forecast_df = pd.read_csv("subsystem1_product_recommendations_v2.csv")
trend_df = pd.read_csv("category_social_trends.csv")
risk_df = pd.read_csv("risk_scoring_output_v2.csv")

In [3]:
trend_df = trend_df.rename(columns={
    "social_trend_score": "trend_score"
})

In [4]:
trend_df = trend_df[[
    "CategoryName",
    "trend_score"
]]

In [5]:
forecast_df["confidence_score"] = forecast_df["probability_up"]

In [6]:
print(forecast_df.columns.tolist())


['CategoryName', 'ProductName', 'product_share', 'product_qty', 'category_qty', 'probability_up', 'growth_signal', 'category_adjusted_forecast', 'base_reorder_qty', 'confidence_score', 'forecast_score']


In [7]:
forecast_keep = forecast_df[[
    "ProductName",
    "CategoryName",
    "forecast_score",
    "base_reorder_qty",
    "confidence_score"
]]

In [10]:
trend_df = trend_df.rename(columns={"social_trend_score": "trend_score"})
trend_keep = trend_df[["CategoryName", "trend_score"]]

In [8]:
risk_keep = risk_df[[
    "ProductName",
    "risk_score",
    "risk_level",
    "risk_drivers"
]]

In [11]:
fusion_df = forecast_keep.merge(trend_keep, on="CategoryName", how="left")
fusion_df = fusion_df.merge(risk_keep, on="ProductName", how="left")

In [12]:
# Ensure fallback works
fusion_df["base_reorder_qty"] = fusion_df["base_reorder_qty"].fillna(0)
fusion_df["trend_score"] = fusion_df["trend_score"].fillna(0)
fusion_df["risk_score"] = fusion_df["risk_score"].fillna(0)


In [13]:
wf = 0.5
wt = 0.3
wr = 0.2

fusion_df["final_score"] = (
    wf * fusion_df["forecast_score"] +
    wt * fusion_df["trend_score"] +
    wr * fusion_df["risk_score"]
).round(3)

In [14]:
alpha = 0.3
beta = 0.2

fusion_df["final_recommended_qty"] = (
    fusion_df["base_reorder_qty"] *
    (1 + alpha * fusion_df["trend_score"] + beta * fusion_df["risk_score"])
).round(0).clip(lower=0)

In [16]:
fusion_export = fusion_df[[
    "CategoryName",
    "ProductName",
    "base_reorder_qty",
    "forecast_score",
    "trend_score",
    "risk_score",
    "final_score",
    "final_recommended_qty",
    "risk_level",
    "risk_drivers",
    "confidence_score"
]]

fusion_export.to_csv("subsystem4_final_recommendations_v2.csv", index=False)

print("Saved: subsystem4_final_recommendations_v2.csv")

Saved: subsystem4_final_recommendations_v2.csv
